# 03b — RM-b: Frozen Encoder + Feature Extraction

Skenario **RM-b**: encoder `indobert-base-p2` **dibekukan**, hanya **classification head** yang dilatih. Ini **training #2** dari 2 training IndoBERT.

Strategi: **ekstrak embedding sekali** (encoder beku, mean-pooling) → simpan → latih head linear di atas embedding. **Embedding train disimpan & dipakai ulang oleh RM-c (RAC)** — jadi encoder beku hanya "diproses" sekali untuk 2 skenario.

**Dijalankan di Google Colab** (GPU). Layout Drive sama seperti `03a` (lihat notebook itu).

## 1. Setup Colab

In [2]:
import os, sys, random, json, time
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Bukan Colab — pakai path lokal.')

if IN_COLAB:
    os.system('pip install -q -U transformers scikit-learn')

PROJECT_DIR = Path('/content/drive/MyDrive/IndoBERT-with-RAC') if IN_COLAB else Path('..')
SRC_DIR = PROJECT_DIR / 'src'
assert SRC_DIR.exists(), f'src tidak ditemukan di {SRC_DIR} — cek PROJECT_DIR'
sys.path.insert(0, str(SRC_DIR))

import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Mounted at /content/drive
device: cuda | Tesla T4


## 2. Konfigurasi (hyperparameter RM-b)

In [3]:
CFG = dict(
    scenario     = 'RM-b',
    model_name   = 'indobenchmark/indobert-base-p2',
    max_length   = 128,
    epochs       = 5,
    batch_size   = 32,        # untuk ekstraksi fitur & training head
    lr           = 2e-4,
    weight_decay = 0.01,
    dropout      = 0.1,
    use_amp      = True,
    seed         = 42,
)
DATA_DIR  = PROJECT_DIR / 'dataset' / 'splits'
META_PATH = PROJECT_DIR / 'dataset' / 'processed' / 'metadata.json'
CKPT_DIR  = PROJECT_DIR / 'results' / 'checkpoints' / 'rmb'
FEAT_DIR  = PROJECT_DIR / 'results' / 'features'
MET_DIR   = PROJECT_DIR / 'results' / 'metrics'
FIG_DIR   = PROJECT_DIR / 'results' / 'figures'
for d in (CKPT_DIR, FEAT_DIR, MET_DIR, FIG_DIR): d.mkdir(parents=True, exist_ok=True)
print(json.dumps(CFG, indent=2))

{
  "scenario": "RM-b",
  "model_name": "indobenchmark/indobert-base-p2",
  "max_length": 128,
  "epochs": 5,
  "batch_size": 32,
  "lr": 0.0002,
  "weight_decay": 0.01,
  "dropout": 0.1,
  "use_amp": true,
  "seed": 42
}


## 3. Data, Tokenizer, DataLoader

In [4]:
import pandas as pd
from torch.utils.data import DataLoader
from dataset import load_tokenizer, GamblingCommentDataset

train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')
print('train/val/test:', len(train_df), len(val_df), len(test_df))

tokenizer = load_tokenizer(CFG['model_name'])

def make_loader(df):
    ds = GamblingCommentDataset(df['text_clean'], df['label'], tokenizer=tokenizer,
                                max_length=CFG['max_length'])
    return DataLoader(ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

loaders = {'train': make_loader(train_df), 'val': make_loader(val_df), 'test': make_loader(test_df)}

class_weights = json.load(open(META_PATH, encoding='utf-8'))['class_weights']
weight = torch.tensor([class_weights['0'], class_weights['1']], dtype=torch.float, device=device)
print('class weights:', weight.tolist())

train/val/test: 6588 1402 1405


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

class weights: [0.6110183596611023, 2.7518796920776367]


## 4. Ekstraksi Fitur (encoder beku, mean-pooling) + Simpan

Embedding disimpan ke `results/features/` — **dipakai ulang oleh RM-c**. Encoder tidak dilatih (0 trainable param).

In [5]:
import modeling as M
import evaluate as E

encoder = M.build_encoder(tokenizer, model_name=CFG['model_name']).to(device)
print('encoder params:', E.count_parameters(encoder))   # trainable = 0

features = {}
t_ext = time.perf_counter()
if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()
for split, loader in loaders.items():
    emb, lab = M.extract_features(encoder, loader, device, use_amp=CFG['use_amp'])
    features[split] = (emb, lab)
    np.save(FEAT_DIR / f'{split}_emb.npy', emb)
    np.save(FEAT_DIR / f'{split}_label.npy', lab)
    print(f'  {split}: emb {emb.shape} -> disimpan')
extract_time = time.perf_counter() - t_ext
extract_peak_mem = E.peak_gpu_mem_mb()
print(f'Waktu ekstraksi fitur: {extract_time:.0f}s | peak GPU mem: {extract_peak_mem:.0f} MB')

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

encoder params: {'total_params': 109483776, 'trainable_params': 0, 'trainable_pct': 0.0}


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

/content/drive/MyDrive/IndoBERT-with-RAC/src/modeling.py:108: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):


  train: emb (6588, 768) -> disimpan
  val: emb (1402, 768) -> disimpan
  test: emb (1405, 768) -> disimpan
Waktu ekstraksi fitur: 21s | peak GPU mem: 531 MB


## 5. Latih Head (linear probe) di atas Fitur Beku

In [6]:
import torch.nn as nn

Xtr = torch.tensor(features['train'][0], device=device); ytr = torch.tensor(features['train'][1], device=device)
Xva = torch.tensor(features['val'][0], device=device);   yva = torch.tensor(features['val'][1], device=device)

head = M.FrozenHead(hidden_size=Xtr.shape[1], num_labels=2, dropout=CFG['dropout']).to(device)
print('head params:', E.count_parameters(head))

criterion = nn.CrossEntropyLoss(weight=weight)
optimizer = torch.optim.AdamW(head.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])

# training head: full-batch pada embedding (dataset kecil, muat di memori)
from torch.utils.data import TensorDataset, DataLoader as DL
head_loader = DL(TensorDataset(Xtr, ytr), batch_size=CFG['batch_size'], shuffle=True)

def eval_head(X, y):
    head.eval()
    with torch.no_grad():
        pred = head(X).argmax(1).cpu().numpy()
    return E.classification_metrics(y.cpu().numpy(), pred)

history, best_f1 = [], -1.0
t_train = time.perf_counter()
for epoch in range(1, CFG['epochs'] + 1):
    head.train(); running = 0.0
    for xb, yb in head_loader:
        optimizer.zero_grad()
        loss = criterion(head(xb), yb)
        loss.backward(); optimizer.step()
        running += loss.item() * yb.size(0)
    vm = eval_head(Xva, yva)
    history.append({'epoch': epoch, 'train_loss': running/len(ytr), 'val_f1_macro': vm['f1_macro'], 'val_acc': vm['accuracy']})
    print(f"epoch {epoch}/{CFG['epochs']} | loss {running/len(ytr):.4f} | val F1-macro {vm['f1_macro']:.4f} | acc {vm['accuracy']:.4f}")
    if vm['f1_macro'] > best_f1:
        best_f1 = vm['f1_macro']
        torch.save({'head_state': head.state_dict(), 'config': CFG, 'val_f1_macro': best_f1,
                    'epoch': epoch, 'hidden_size': Xtr.shape[1]}, CKPT_DIR / 'head_best.pt')
head_train_time = time.perf_counter() - t_train
print(f'\nWaktu training head: {head_train_time:.1f}s | best val F1-macro: {best_f1:.4f}')

head params: {'total_params': 1538, 'trainable_params': 1538, 'trainable_pct': 100.0}
epoch 1/5 | loss 0.4081 | val F1-macro 0.8754 | acc 0.9230
epoch 2/5 | loss 0.2657 | val F1-macro 0.8932 | acc 0.9315
epoch 3/5 | loss 0.2261 | val F1-macro 0.9033 | acc 0.9379
epoch 4/5 | loss 0.2076 | val F1-macro 0.9012 | acc 0.9358
epoch 5/5 | loss 0.1942 | val F1-macro 0.9063 | acc 0.9394

Waktu training head: 3.1s | best val F1-macro: 0.9063


## 6. Evaluasi Test + Efisiensi + Simpan

In [7]:
ckpt = torch.load(CKPT_DIR / 'head_best.pt', map_location=device)
head.load_state_dict(ckpt['head_state']); head.eval()

Xte = torch.tensor(features['test'][0], device=device); yte = features['test'][1]
with torch.no_grad():
    y_pred = head(Xte).argmax(1).cpu().numpy()
y_true = yte

metrics = E.classification_metrics(y_true, y_pred)
print('=== Test metrics (RM-b) ===')
for k in ['accuracy','f1_macro','precision_macro','recall_macro','f1_weighted','f1_class1']:
    print(f'  {k:18s}: {metrics[k]:.4f}')

E.plot_confusion_matrix(y_true, y_pred, FIG_DIR / 'rmb_confusion.png', title='RM-b — Confusion Matrix')

# efisiensi: trainable = head; latency = encoder(1 sampel) + head
head_params = E.count_parameters(head)
total_train_time = extract_time + head_train_time    # ekstraksi (sekali) + latih head

one = next(iter(loaders['test']))
ids1 = one['input_ids'][:1].to(device); attn1 = one['attention_mask'][:1].to(device)
tti1 = one.get('token_type_ids'); tti1 = tti1[:1].to(device) if tti1 is not None else None
from torch.cuda.amp import autocast
def _predict(_):
    with torch.no_grad(), autocast(enabled=CFG['use_amp'] and device.type=='cuda'):
        out = encoder(input_ids=ids1, attention_mask=attn1, token_type_ids=tti1)
        feat = M.mean_pool(out.last_hidden_state, attn1).float()
        return head(feat)
latency = E.measure_latency(_predict, None)

row = {'scenario': 'RM-b', **{k: round(v,6) for k,v in metrics.items()},
       'trainable_params': head_params['trainable_params'],
       'encoder_params': E.count_parameters(encoder)['total_params'],
       'extract_time_s': round(extract_time,1), 'head_train_time_s': round(head_train_time,1),
       'total_train_time_s': round(total_train_time,1),
       'extract_peak_gpu_mem_mb': round(extract_peak_mem,1),
       'latency_ms_per_sample': round(latency,4), 'best_val_f1_macro': round(best_f1,6),
       **{f'hp_{k}': v for k,v in CFG.items()}}
E.save_metrics_csv(row, MET_DIR / 'rmb_metrics.csv')
pd.DataFrame(history).to_csv(MET_DIR / 'rmb_history.csv', index=False)
print('\nEfisiensi:', {k: row[k] for k in ['trainable_params','total_train_time_s','extract_peak_gpu_mem_mb','latency_ms_per_sample']})
print('Tersimpan: rmb_metrics.csv, rmb_history.csv, rmb_confusion.png, checkpoints/rmb/head_best.pt, features/*.npy')

=== Test metrics (RM-b) ===
  accuracy          : 0.9367
  f1_macro          : 0.9006
  precision_macro   : 0.8783
  recall_macro      : 0.9294
  f1_weighted       : 0.9387
  f1_class1         : 0.8408


/tmp/ipykernel_713/1983528152.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(enabled=CFG['use_amp'] and device.type=='cuda'):



Efisiensi: {'trainable_params': 1538, 'total_train_time_s': 23.9, 'extract_peak_gpu_mem_mb': 530.5, 'latency_ms_per_sample': 51.6103}
Tersimpan: rmb_metrics.csv, rmb_history.csv, rmb_confusion.png, checkpoints/rmb/head_best.pt, features/*.npy


## Ringkasan

RM-b (frozen encoder + head) selesai — **training #2** dari 2.

- Trainable params **hanya head** (~1,5k) vs ~125M RM-a → reduksi >99,9% (mendukung kriteria efisiensi).
- Embedding beku tersimpan di `results/features/{train,val,test}_emb.npy` → **dipakai ulang RM-c** (FAISS index dari `train_emb.npy`).
- **Langkah berikut**: `03c_rmc_rac.ipynb` — retrieval-augmented (tanpa training baru): fusi logit head + k-NN FAISS.